# 3 · A model shaped for the browser

`2_upscale_nn.ipynb` showed that a learned model beats the filters. This notebook makes
it **fast in the browser**. The changes, biggest win first:

1. **No `Resize`.** Bicubic x2 is a fixed convolution, so we write it as one
   (`models.BicubicSkip`). Same result, and the browser runs `Conv` fast but `Resize` slowly.
2. **Add at low resolution.** Both branches are added *before* the `PixelShuffle`, so
   there is one shuffle and a quarter of the pixels to add.
3. **Narrower.** A 32-16 body instead of 64-32: a third of the work for about 1 dB.
4. **float16.** Half the size, same quality, faster on GPUs that support it.

Two tricks that did *not* help (int8, depthwise convolutions), and how the browser page
runs the model (GPU-resident tensors, graph capture), are in `summary.md`,
"The browser-shaped model".

Needs: `0_data_preprocess.ipynb`. Writes `checkpoints/upscale_web.pt`, which
`5_upscale_geometry.ipynb` uses.

In [ ]:
"""Settings and data."""

import numpy as np
import torch
from PIL import Image, ImageDraw

from helpers import metrics, models, patches, speed, training, webexport
from helpers.training import DATA_DIR, DEVICE

CHECKPOINT = training.CHECKPOINT_DIR / "upscale_web.pt"
FRAME_DIR = DATA_DIR / "synthetic"

WIDTHS = (32, 16)    # channels of the body's convolutions
STEM = 5             # size of the first kernel
SKIP = 5             # size of the bicubic skip kernel (bicubic x2 needs 4 taps; 5 is the smallest odd size)
EPOCHS = 100
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
SEED = 0
DEPLOY_STEP = 256    # a bigger tile, also published (fewer runs per frame)

training.seed_everything(SEED)
manifest = training.load_manifest()
training.describe(manifest)
SCALE = manifest["scale"]
train_loader, val_loader = training.loaders(DATA_DIR, manifest, BATCH_SIZE)
trim = training.trimmer(manifest)

## 1. The skip is a convolution

Bicubic x2 is linear and shift-invariant, which is exactly what a convolution is: each of
the 4 output sub-pixels is a fixed small filter of the input. `models.bicubic_kernel`
reads those filters out of `F.interpolate`, and we check the result matches it.

In [ ]:
difference = models.check_skip_is_bicubic(SCALE, SKIP)
print(f"conv skip vs F.interpolate: {difference:.1e} apart (float32 rounding)")

model = models.PlainUpscaler(WIDTHS, STEM, SKIP, SCALE).to(DEVICE)
print(model)
print(f"{models.count_parameters(model):,} parameters, {models.count_parameters(model, learned_only=True):,} learned")
print(f"{speed.macs_per_lr_pixel(model):,.0f} multiply-adds per LR pixel, halo {model.halo}")

### What `Resize` costs

The same model exported twice - once with the skip as `F.interpolate` (a `Resize` node)
and once as the convolution - and timed on the desktop CPU.

In [ ]:
class ResizeSkip(models.PlainUpscaler):
    """The same model with the skip as F.interpolate - what 2_upscale_nn.ipynb does."""

    def forward(self, x):
        return models.bicubic(x, self.scale) + self.shuffle(self.residual(x))


size = webexport.TILE_STEP + 2 * model.halo
comparison_dir = training.CHECKPOINT_DIR / "comparison"
for name, variant in {"conv skip": model, "Resize skip": ResizeSkip(WIDTHS, STEM, SKIP, SCALE)}.items():
    info = webexport.export(variant, f"skip_{name.split()[0].lower()}.onnx", size, models_dir=comparison_dir)
    print(f"{name:<12}", webexport.summary_line(info, webexport.time_graph(info["path"], size)))

## 2. Train

Same recipe as `2_upscale_nn.ipynb`, but 100 epochs instead of 12 (12 was far from
finished). The model must start exactly at bicubic: the skip is bicubic and the tail
starts at zero.

In [ ]:
baseline = training.evaluate(models.Resize(SCALE).to(DEVICE), val_loader, trim)
start = training.evaluate(model, val_loader, trim)
assert abs(start["rgb"] - baseline["rgb"]) < 1e-3, "the model should start as bicubic"

history, best = training.train(model, train_loader, val_loader, trim, EPOCHS,
                               learning_rate=LEARNING_RATE, select="rgb")
training.plot_history(history, baseline, key="rgb")

final = training.evaluate(model, val_loader, trim)
metrics.print_table({"bicubic": baseline, "untrained": start, "trained": final})

In [ ]:
lr, hr = next(iter(val_loader))
lr, hr = lr[:4].to(DEVICE), hr[:4].to(DEVICE)
with torch.no_grad():
    training.show_patches({"bicubic": trim(models.bicubic(lr, SCALE)),
                           "model": trim(model.eval()(lr)), "target": hr})

## 3. Save and reload

In [ ]:
training.save_checkpoint(CHECKPOINT, model, architecture="PlainUpscaler", widths=list(WIDTHS),
                         stem=STEM, skip=SKIP, scale=SCALE, halo=model.halo, epochs=EPOCHS,
                         best_epoch=best["epoch"], val=final, val_bicubic=baseline,
                         degradation=manifest["degradation"])

checkpoint = training.load_checkpoint(CHECKPOINT)
loaded = models.PlainUpscaler(tuple(checkpoint["widths"]), checkpoint["stem"],
                              checkpoint["skip"], checkpoint["scale"]).to(DEVICE)
loaded.load_state_dict(checkpoint["state_dict"])
training.assert_same_model(model, loaded)
print("reloaded model is identical")

In [ ]:
"""A whole frame through the loaded model."""

truth = Image.open(sorted(FRAME_DIR.glob("*.png"))[0]).convert("RGB")
truth = truth.crop((0, 0, truth.width - truth.width % SCALE, truth.height - truth.height % SCALE))
small = truth.resize((truth.width // SCALE, truth.height // SCALE), Image.BICUBIC)
with torch.no_grad():
    upscaled = loaded.eval()(training.to_tensor(small))
target = training.to_tensor(truth)
metrics.print_table({"bicubic": metrics.report(models.bicubic(training.to_tensor(small), SCALE), target),
                     "model": metrics.report(upscaled, target)})
Image.fromarray(training.to_image(upscaled)).save(training.SAMPLE_DIR / "upscaled_web.png")

## 4. Export

Four graphs are published, each checked against the torch model first:

- **float32** at the page's 128 step - comparable with every other model on the page;
- **float16** - half the size; the desktop CPU has no float16 kernels, so it is only
  faster on a GPU;
- **int8** - smaller again, but its extra nodes are outside the browser's fast operator
  set, so it is much *slower* in the browser. Published to show that;
- **float16 at step 256** - a bigger tile means fewer runs per frame.

In [ ]:
halo = checkpoint["halo"]
size = webexport.TILE_STEP + 2 * halo
label = f"web {'-'.join(str(w) for w in WIDTHS)}"
val_lr, _ = patches.load(DATA_DIR, "val", manifest)
train_lr, _ = patches.load(DATA_DIR, "train", manifest, count=48)


def tile_from(patch, edge):
    """A real uint8 patch as a float (1, 3, edge, edge) array, its border repeated to fill."""
    x = patch.transpose(2, 0, 1)[None].astype(np.float32) / 255.0
    return np.pad(x, ((0, 0), (0, 0), (0, edge - patch.shape[0]), (0, edge - patch.shape[1])), mode="edge")


noise = np.random.rand(1, 3, size, size).astype(np.float32)

fp32 = webexport.export(loaded, f"upscale_web_tile{webexport.TILE_STEP}.onnx", size,
                        label=f"{label} fp32", halo=halo)
webexport.check_graph(fp32["path"], loaded, noise)

fp16 = webexport.to_float16(fp32, f"upscale_web_fp16_tile{webexport.TILE_STEP}.onnx")
webexport.write_metadata(fp16["path"], {"label": f"{label} fp16"})
webexport.check_graph(fp16["path"], loaded, noise, np.float16, tolerance=4e-3)

# int8 learns its value ranges from real patches, so it is also checked on a real patch
int8 = webexport.to_int8(fp32, f"upscale_web_int8_tile{webexport.TILE_STEP}.onnx",
                         [tile_from(p, size) for p in train_lr])
webexport.write_metadata(int8["path"], {"label": f"{label} int8 (Q/DQ, outside the budget)"})
webexport.check_graph(int8["path"], loaded, tile_from(val_lr[0], size), tolerance=2e-2)

deploy_size = DEPLOY_STEP + 2 * halo
deploy = webexport.export(loaded, f"upscale_webbig_tile{DEPLOY_STEP}.onnx", deploy_size, halo=halo)
deploy = webexport.to_float16(deploy, f"upscale_webbig_tile{DEPLOY_STEP}.onnx")
webexport.write_metadata(deploy["path"], {"label": f"{label} fp16, step {DEPLOY_STEP}"})
webexport.check_graph(deploy["path"], loaded, np.random.rand(1, 3, deploy_size, deploy_size).astype(np.float32),
                      np.float16, tolerance=4e-3)

for info, shape, dtype in [(fp32, size, np.float32), (fp16, size, np.float16),
                           (int8, size, np.float32), (deploy, deploy_size, np.float16)]:
    ms = webexport.time_graph(info["path"], shape, dtype)
    webexport.annotate(info, cpu_ms=round(ms, 2))
    print(webexport.summary_line(info, ms))

In [ ]:
"""Quality of each precision, measured by running the ONNX graph itself."""

eval_dir = training.CHECKPOINT_DIR / "evaluation"
any_size = webexport.export(loaded, "web_dynamic.onnx", manifest["input_lr"], halo=halo,
                            dynamic=True, models_dir=eval_dir)
versions = {
    "float32": (any_size, np.float32),
    "float16": (webexport.to_float16(any_size, "web_dynamic_fp16.onnx", models_dir=eval_dir), np.float16),
    "int8": (webexport.to_int8(any_size, "web_dynamic_int8.onnx",
                               [tile_from(p, p.shape[0]) for p in train_lr], models_dir=eval_dir), np.float32),
}
results = {name: training.evaluate(webexport.OnnxModel(info["path"], dtype), val_loader, trim)
           for name, (info, dtype) in versions.items()}
metrics.print_table(results)

## 5. Skip the tiles that did not change

The biggest saving is not in the model at all: on a desktop, most of the screen is the
same as in the previous frame. A tile whose input did not change has the same output,
so it does not need to run. Below, on a drawn desktop with a dragged window, a moving
cursor and a playing video: how many tiles change per frame.

In [ ]:
def desktop_sequence(count, size=(1920, 1080), seed=SEED):
    """Frames of a drawn desktop: still windows, one dragged window, a cursor, a video."""
    rng = np.random.default_rng(seed)
    width, height = size
    background = Image.new("RGB", size, (32, 36, 44))
    draw = ImageDraw.Draw(background)
    for y in range(height):
        draw.line([(0, y), (width, y)], fill=(32 + y // 40, 36 + y // 40, 44 + y // 40))
    for index in range(3):
        x, y = 60 + index * 520, 80 + index * 90
        draw.rectangle([x, y, x + 460, y + 620], fill=(238, 238, 240), outline=(90, 110, 160), width=2)
        draw.rectangle([x, y, x + 460, y + 30], fill=(90, 110, 160))
        for row in range(y + 50, y + 600, 16):
            draw.rectangle([x + 16, row, x + 16 + int(rng.integers(120, 420)), row + 5], fill=(40, 40, 48))

    frames = []
    for index in range(count):
        frame = background.copy()
        drawing = ImageDraw.Draw(frame)
        x, y = 1180 + 6 * index, 420 + 3 * index                     # dragged window
        drawing.rectangle([x, y, x + 520, y + 380], fill=(250, 250, 252), outline=(160, 90, 90), width=2)
        drawing.rectangle([x, y, x + 520, y + 28], fill=(160, 90, 90))
        video = rng.integers(0, 255, (180, 320, 3), dtype=np.uint8)  # video playing
        frame.paste(Image.fromarray(video).resize((640, 360), Image.NEAREST), (200, 620))
        cx, cy = 900 + 11 * index, 300 + 7 * index                   # cursor
        drawing.polygon([(cx, cy), (cx + 14, cy + 40), (cx + 22, cy + 22)], fill=(255, 255, 255))
        frames.append(frame)
    return frames


def changed_tiles(frames, step):
    """Average share of LR tiles that differ from the previous frame."""
    low = [np.asarray(f.resize((f.width // SCALE, f.height // SCALE), Image.BICUBIC)) for f in frames]
    rows, cols = -(-low[0].shape[0] // step), -(-low[0].shape[1] // step)
    shares = []
    for before, now in zip(low, low[1:]):
        changed = sum(not np.array_equal(now[r * step:(r + 1) * step, c * step:(c + 1) * step],
                                         before[r * step:(r + 1) * step, c * step:(c + 1) * step])
                      for r in range(rows) for c in range(cols))
        shares.append(changed / (rows * cols))
    return float(np.mean(shares)), rows * cols


frames = desktop_sequence(24)
BROWSER_MS = {128: 0.190, 256: 0.424}   # ms per tile, measured in the browser (summary.md)
print(f"{'step':>5}{'tiles':>7}{'changed':>9}{'all tiles':>11}{'changed only':>14}")
for step, ms in BROWSER_MS.items():
    share, tiles = changed_tiles(frames, step)
    print(f"{step:5}{tiles:7}{share:9.0%}{tiles * ms:8.1f} ms{tiles * share * ms:11.1f} ms")